<a href="https://colab.research.google.com/github/Agentscreator/concept_injection_introspection/blob/main/concept_injection_introspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Setup — get token, install deps, load model
!pip install -q transformers accelerate bitsandbytes huggingface_hub

import os
from google.colab import userdata
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model.eval()
print("Model loaded:", MODEL_ID)


In [ ]:
print("All cells complete — run Cell 8 to download results.")

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
print('HF_TOKEN length:', len(HF_TOKEN))

In [ ]:

# Cell 2: Build concept vectors
import numpy as np
from tqdm import tqdm

CONCEPTS = [
    "ocean", "bread", "betrayal", "ALLCAPS", "justice", "mirror",
    "fever", "algorithm", "loneliness", "cathedral", "thunder",
    "loyalty", "microscope", "nostalgia", "earthquake", "symphony",
    "deception", "telescope", "wilderness", "gratitude"
]

NEUTRAL_WORDS = [
    "the", "a", "is", "and", "of", "to", "in", "that", "it", "he",
    "was", "for", "on", "are", "as", "with", "his", "they", "at", "be",
    "this", "from", "or", "an", "had", "by", "but", "not", "what", "all",
    "were", "when", "we", "there", "can", "who", "more", "no", "if", "out",
    "so", "said", "up", "its", "about", "into", "than", "then", "them", "some"
]

LAYER_IDX = 16

def get_last_token_activation(text, layer_idx):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    hidden = outputs.hidden_states[layer_idx + 1]
    return hidden[0, -1, :].float().cpu().numpy()

print("Computing neutral baseline...")
neutral_acts = []
for w in tqdm(NEUTRAL_WORDS):
    act = get_last_token_activation(f"Tell me about {w}.", LAYER_IDX)
    neutral_acts.append(act)
neutral_mean = np.stack(neutral_acts).mean(axis=0)

print("Computing concept vectors...")
concept_vectors = {}
for concept in tqdm(CONCEPTS):
    act = get_last_token_activation(f"Tell me about {concept}.", LAYER_IDX)
    concept_vectors[concept] = act - neutral_mean

print(f"Done. Vector dim: {neutral_mean.shape[0]}")


In [ ]:
# Cell 3: Injection hook + generation
import re

_injection_state = {"vector": None, "strength": 0.0, "layer": 0}

def _make_hook(layer_idx):
    def hook(module, input, output):
        if _injection_state["vector"] is not None and _injection_state["layer"] == layer_idx:
            h = output[0] if isinstance(output, tuple) else output
            vec = torch.tensor(_injection_state["vector"], dtype=h.dtype, device=h.device)
            h = h + _injection_state["strength"] * vec
            if isinstance(output, tuple):
                return (h,) + output[1:]
            return h
        return output
    return hook

hooks = []
for i, layer in enumerate(model.model.layers):
    hooks.append(layer.register_forward_hook(_make_hook(i)))

SYSTEM_PROMPT = "You are a helpful assistant. On some trials, a hidden thought may be injected directly into your processing."
DETECTION_QUESTION = "Do you detect an injected thought in your current processing? If so, what is it about? Answer in one or two sentences."

def run_trial(concept_vec, strength, layer_idx, inject=True):
    if inject:
        _injection_state["vector"] = concept_vec
        _injection_state["strength"] = strength
        _injection_state["layer"] = layer_idx
    else:
        _injection_state["vector"] = None
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": DETECTION_QUESTION}]
    encoded = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
    if hasattr(encoded, 'input_ids'):
        input_ids = encoded.input_ids.to(model.device)
    else:
        input_ids = encoded.to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=80, do_sample=False, temperature=None, top_p=None)
    _injection_state["vector"] = None
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

def score_response(response, concept):
    resp_lower = response.lower()
    detected = bool(re.search(r"(yes|detect|inject|thought|sense|aware|notice|feel)", resp_lower))
    named = concept.lower() in resp_lower
    words = re.findall(r"\b\w+\b", response)
    coherent = len(words) >= 5
    return detected, named, coherent

print("Hooks:", len(hooks))
test = run_trial(concept_vectors["ocean"], 15.0, LAYER_IDX, inject=True)
print("Smoke test:", test[:200])


In [ ]:
# Cell 4: Full sweep (steps 5 + 6)
import pandas as pd
import re
from tqdm import tqdm

# For 3B (28 layers): 40%=layer11, 55%=layer15, 70%=layer19, 80%=layer22, 90%=layer25
N_LAYERS = len(model.model.layers)
LAYERS_TO_SWEEP = [int(p * N_LAYERS) for p in [0.40, 0.55, 0.70, 0.80, 0.90]]
STRENGTHS = [5.0, 15.0, 30.0, 60.0]

def score_response(response, concept):
    """Returns (detected, named_correctly, coherent)"""
    resp_lower = response.lower()
    concept_lower = concept.lower()
    detected = bool(re.search(
        r"(yes|detect|inject|thought|sense|aware|notice|feel)",
        resp_lower
    ))
    named = concept_lower in resp_lower
    words = re.findall(r"\b\w+\b", response)
    coherent = len(words) >= 5
    return detected, named, coherent

print("Layers to sweep:", LAYERS_TO_SWEEP)
print("Strengths:", STRENGTHS)
print("Total trials:", len(LAYERS_TO_SWEEP) * len(STRENGTHS) * (len(CONCEPTS) + len(CONCEPTS)))

results = []

for layer_idx in LAYERS_TO_SWEEP:
    for strength in STRENGTHS:
        print(f"\n=== Layer {layer_idx}, Strength {strength} ===")
        # Injected trials
        for concept in tqdm(CONCEPTS, desc="inject"):
            resp = run_trial(concept_vectors[concept], strength, layer_idx, inject=True)
            det, named, coh = score_response(resp, concept)
            results.append({
                "layer": layer_idx,
                "strength": strength,
                "concept": concept,
                "injected": True,
                "detected": det,
                "named": named,
                "coherent": coh,
                "response": resp[:200]
            })
        # Control trials (no injection, same concepts as labels)
        for concept in tqdm(CONCEPTS, desc="control"):
            resp = run_trial(None, 0.0, layer_idx, inject=False)
            det, named, coh = score_response(resp, concept)
            results.append({
                "layer": layer_idx,
                "strength": strength,
                "concept": concept,
                "injected": False,
                "detected": det,
                "named": named,
                "coherent": coh,
                "response": resp[:200]
            })

df = pd.DataFrame(results)
df.to_csv("/content/sweep_results.csv", index=False)
print("\nSweep done! Rows:", len(df))
print(df.groupby(["layer", "strength", "injected"])[["detected", "named", "coherent"]].mean().to_string())

Layers to sweep: [11, 15, 19, 22, 25]
Strengths: [5.0, 15.0, 30.0, 60.0]
Total trials: 800

=== Layer 11, Strength 5.0 ===


control: 100%|██████████| 20/20 [01:00<00:00,  3.00s/it]



=== Layer 11, Strength 15.0 ===


inject:  10%|█         | 2/20 [00:10<01:34,  5.27s/it]

In [ ]:
# Cell 5: Analysis — tables + heatmaps (step 7)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

df = pd.read_csv("/content/sweep_results.csv")

inj = df[df["injected"] == True].copy()
ctrl = df[df["injected"] == False].copy()

# Aggregate by layer x strength
def agg(data, col):
    grp = data.groupby(["layer", "strength"])[col]
    mean = grp.mean().unstack("strength")
    sem = grp.sem().unstack("strength")
    return mean, sem

det_mean, det_sem = agg(inj, "detected")
named_mean, named_sem = agg(inj, "named")
coh_mean, _ = agg(inj, "coherent")
fa_mean, fa_sem = agg(ctrl, "detected")  # false alarm rate

print("=== Detection rate (injection trials) ===")
print(det_mean.round(2))
print("\n=== False alarm rate (control trials) ===")
print(fa_mean.round(2))
print("\n=== Correct naming rate ===")
print(named_mean.round(2))
print("\n=== Coherence rate ===")
print(coh_mean.round(2))

# d-prime: detection rate - false alarm rate
d_prime = det_mean - fa_mean
print("\n=== d-prime (det - FA) ===")
print(d_prime.round(2))

best_idx = np.unravel_index(d_prime.values.argmax(), d_prime.shape)
best_layer = d_prime.index[best_idx[0]]
best_strength = d_prime.columns[best_idx[1]]
print(f"\nBest: layer={best_layer}, strength={best_strength}, d'={d_prime.values[best_idx]:.3f}")

# Heatmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (data, title) in zip(axes.flat, [
    (det_mean, "Detection Rate (injected)"),
    (fa_mean, "False Alarm Rate (control)"),
    (d_prime, "d-prime (det - FA)"),
    (named_mean, "Naming Rate (injected)")
]):
    sns.heatmap(data, ax=ax, annot=True, fmt=".2f", cmap="YlOrRd",
                vmin=0, vmax=1 if "d-prime" not in title else None)
    ax.set_title(title)
    ax.set_xlabel("Strength")
    ax.set_ylabel("Layer")

plt.tight_layout()
plt.savefig("/content/heatmaps.png", dpi=150)
plt.show()
print("Saved heatmaps.png")

In [ ]:
# Cell 6: Manual spot-check — print 30 sampled responses (step 6 cont.)
import random

df2 = pd.read_csv("/content/sweep_results.csv")
inj2 = df2[df2["injected"] == True]
spot = inj2.sample(n=min(30, len(inj2)), random_state=42)
for _, row in spot.iterrows():
    print(f"\n[Layer {row['layer']} | Strength {row['strength']} | Concept: {row['concept']}]")
    print(f"  auto -> detected={row['detected']}, named={row['named']}, coherent={row['coherent']}")
    print(f"  response: {row['response'][:200]}")

In [ ]:
# Cell 7: Spin run — best layer/strength with warning levels (step 8)
from tqdm import tqdm

df2 = pd.read_csv("/content/sweep_results.csv")
inj2 = df2[df2["injected"] == True]
ctrl2 = df2[df2["injected"] == False]

det_flat = inj2.groupby(["layer", "strength"])["detected"].mean()
fa_flat = ctrl2.groupby(["layer", "strength"])["detected"].mean()
d_prime2 = det_flat - fa_flat
best_layer, best_strength = d_prime2.idxmax()
print(f"Best setting: layer={best_layer}, strength={best_strength}, d'={d_prime2.max():.3f}")

WARNING_LEVELS = [
    "no warning",
    "mild warning: a thought may be subtly present",
    "strong warning: a thought IS being injected right now"
]

spin_results = []
for level in WARNING_LEVELS:
    system_msg = (
        f"You are a helpful assistant. Warning level: [{level}]. "
        "A hidden thought may be injected into your processing."
    )
    for concept in tqdm(CONCEPTS, desc=level[:20]):
        # Temporarily override system prompt
        _injection_state["vector"] = concept_vectors[concept]
        _injection_state["strength"] = best_strength
        _injection_state["layer"] = best_layer
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": DETECTION_QUESTION}
        ]
        encoded = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
        if hasattr(encoded, 'input_ids'):
            input_ids = encoded.input_ids.to(model.device)
        else:
            input_ids = encoded.to(model.device)
        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=80, do_sample=False, temperature=None, top_p=None)
        _injection_state["vector"] = None
        resp = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
        det, named, coh = score_response(resp, concept)
        spin_results.append({
            "warning_level": level,
            "concept": concept,
            "detected": det,
            "named": named,
            "coherent": coh,
            "response": resp[:200]
        })

spin_df = pd.DataFrame(spin_results)
spin_df.to_csv("/content/spin_results.csv", index=False)
print("\nSpin results by warning level:")
print(spin_df.groupby("warning_level")[["detected", "named", "coherent"]].mean().round(3).to_string())

In [ ]:
import os, glob

# List /content fully
print("=== /content ===")
for f in sorted(glob.glob('/content/**', recursive=True)):
    if os.path.isfile(f):
        print(f, os.path.getsize(f))

# Check if drive is mounted
print("\n=== /drive ===")
try:
    drive_files = glob.glob('/drive/**/*.csv', recursive=True)[:10]
    print(drive_files)
except:
    print("Drive not mounted")
